
# 04 — Recommendations (Audit-Ready)

Czysty, minimalny notatnik weryfikujący moduł `recommend.py` po naprawach.  
Zawiera wyłącznie niezbędne kroki: ścieżki, import, sanity-check artefaktów, dwa smoke-testy i mini-raport.


In [37]:

# 1) Ścieżki (dopasowane do potwierdzonego układu)
import os
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\annas\projekty_infoshare\projekt ML")
ARTIFACTS    = PROJECT_ROOT / "artifacts"
DATA_DIR     = PROJECT_ROOT / "data"
RECOMMEND_PY = PROJECT_ROOT / "recommend.py"

os.chdir(PROJECT_ROOT)

print("CWD          :", Path.cwd())
print("PROJECT_ROOT :", PROJECT_ROOT)
print("ARTIFACTS    :", ARTIFACTS, "EXISTS:", ARTIFACTS.exists())
print("DATA_DIR     :", DATA_DIR, "EXISTS:", DATA_DIR.exists())
print("recommend.py :", RECOMMEND_PY, "EXISTS:", RECOMMEND_PY.exists())


CWD          : C:\Users\annas\projekty_infoshare\projekt ML
PROJECT_ROOT : C:\Users\annas\projekty_infoshare\projekt ML
ARTIFACTS    : C:\Users\annas\projekty_infoshare\projekt ML\artifacts EXISTS: True
DATA_DIR     : C:\Users\annas\projekty_infoshare\projekt ML\data EXISTS: True
recommend.py : C:\Users\annas\projekty_infoshare\projekt ML\recommend.py EXISTS: True


In [33]:

# 2) Import recommend.py po absolutnej ścieżce (bez sys.path zabiegów)
import importlib.util
from pathlib import Path

RECOMMEND_PY = Path(r"C:\Users\annas\projekty_infoshare\projekt ML\recommend.py")

spec = importlib.util.spec_from_file_location("recommend", str(RECOMMEND_PY))
recommend = importlib.util.module_from_spec(spec)
spec.loader.exec_module(recommend)

print("✅ Import OK:", RECOMMEND_PY)
print("Publiczne atrybuty:", [a for a in dir(recommend) if not a.startswith("_")][:30])


✅ Import OK: C:\Users\annas\projekty_infoshare\projekt ML\recommend.py
Publiczne atrybuty: ['ARTIFACTS', 'DEFAULT_FEATURE_CSV', 'Dict', 'HERE', 'List', 'Path', 'Tuple', 'annotations', 'cosine_similarity', 'joblib', 'json', 'np', 'pd', 'recommend_by_answers', 'recommend_by_plant']


In [34]:

# 3) Sanity check artefaktów i wczytanie feature_list.json (latin-1)
from pathlib import Path
import json

ARTIFACTS = Path(r"C:\Users\annas\projekty_infoshare\projekt ML\artifacts")

required = ["ohe.pkl","scaler.pkl","pca.pkl","kmeans.pkl","feature_list.json"]
missing = [f for f in required if not (ARTIFACTS/f).exists()]
if missing:
    raise FileNotFoundError(f"Brak artefaktów: {missing}")
print("✅ Artefakty obecne:", required)

with open(ARTIFACTS/"feature_list.json","r",encoding="latin-1") as f:
    fl = json.load(f)

if isinstance(fl, dict) and "features_order" in fl:
    feat_order = fl["features_order"]
    cat_cols = [x["name"] for x in feat_order if x.get("type")=="categorical"]
    num_all  = [x["name"] for x in feat_order if x.get("type")=="numeric"]
else:
    cat_cols, num_all = [], []

num_scaled = [c for c in num_all if c != "toxicity_any"]
bin_cols   = ["toxicity_any"] if "toxicity_any" in num_all else []

print("✅ feature_list OK (cat,num):", (len(cat_cols), len(num_all)))
print("cat_cols   :", cat_cols)
print("num_all    :", num_all)
print("num_scaled :", num_scaled)
print("bin_cols   :", bin_cols)


✅ Artefakty obecne: ['ohe.pkl', 'scaler.pkl', 'pca.pkl', 'kmeans.pkl', 'feature_list.json']
✅ feature_list OK (cat,num): (5, 3)
cat_cols   : ['tempmin_pasmo', 'tempmax_pasmo', 'category_group', 'light_level_clean', 'watering_group']
num_all    : ['is_cold_tolerant', 'is_heat_tolerant', 'toxicity_any']
num_scaled : ['is_cold_tolerant', 'is_heat_tolerant']
bin_cols   : ['toxicity_any']


In [35]:

# 4) Smoke-test: recommend_by_answers i recommend_by_plant
import pandas as pd

example_answers = {
    "tempmin_pasmo":     "Tylko temperatura pokojowa",
    "tempmax_pasmo":     "Normalna tolerancja ciepła",
    "category_group":    "Paproć / roślina zielona",
    "light_level_clean": "Low light",
    "watering_group":    "Moderate",
    "is_cold_tolerant":  0,
    "is_heat_tolerant":  0,
    "toxicity_any":      0,
}

df_ans = recommend.recommend_by_answers(example_answers, top_n=5)
print("✅ recommend_by_answers OK. Typ:", type(df_ans))
display(df_ans.head(10))

df_src, *_ = recommend._load_artifacts(recommend.DEFAULT_FEATURE_CSV)
latin_example = df_src["latin"].iloc[0]
print("Źródłowa roślina:", latin_example)

df_plant = recommend.recommend_by_plant(latin_example, top_n=5)
print("✅ recommend_by_plant OK. Typ:", type(df_plant))
display(df_plant.head(10))


✅ recommend_by_answers OK. Typ: <class 'pandas.core.frame.DataFrame'>


,rank,latin,similarity
0,1,Adiantum raddianum,0.831238
1,2,Adiantum hispidulum,0.831238
2,3,Pteris cretica Albo lineata,0.831238
3,4,Pteris cretica Parkeri,0.831238
4,5,Pteris ensiformis Evergemiensis,0.831238


Źródłowa roślina: Aeschynanthus lobianus
✅ recommend_by_plant OK. Typ: <class 'pandas.core.frame.DataFrame'>


,rank,latin,similarity
0,1,Licuala spinosa,0.829332
1,2,Vriesea Ginger,0.801935
2,3,Vriesea splendens,0.801935
3,4,Vriesea Christiane,0.801935
4,5,Tillandsia Creation,0.801935


In [36]:

# 5) Mini-raport ALL GREEN
STATUS = []
def ok(msg): STATUS.append(f"✅ {msg}")
def warn(msg): STATUS.append(f"⚠️ {msg}")
def bad(msg): STATUS.append(f"❌ {msg}")

ok("Artefakty: ohe, scaler, pca, kmeans, feature_list — OK")
ok("feature_list OK (cat/num) — patrz powyżej")

import pandas as pd
if isinstance(df_ans, pd.DataFrame) and {"rank","latin","similarity"}.issubset(df_ans.columns):
    ok("recommend_by_answers → DataFrame (rank/latin/similarity)")
else:
    warn("recommend_by_answers → format inny niż oczekiwany")

if isinstance(df_plant, pd.DataFrame) and {"rank","latin","similarity"}.issubset(df_plant.columns):
    ok("recommend_by_plant → DataFrame (rank/latin/similarity)")
else:
    warn("recommend_by_plant → format inny niż oczekiwany")

print("\n".join(STATUS))


✅ Artefakty: ohe, scaler, pca, kmeans, feature_list — OK
✅ feature_list OK (cat/num) — patrz powyżej
✅ recommend_by_answers → DataFrame (rank/latin/similarity)
✅ recommend_by_plant → DataFrame (rank/latin/similarity)
